In [3]:
# ============================================
# IMPORTS & SETUP (FIXED)
# ============================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.19.0
GPU: []


2026-04-19 18:15:03.495995: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
# ============================================
# DATA PATHS
# ============================================

train_dir = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train"
test_dir  = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/test"

In [5]:
# ============================================
# DATA PIPELINE
# ============================================

IMG_SIZE = 192
BATCH_SIZE = 16   # safer

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# ❗ VALIDATION MUST NOT HAVE AUGMENTATION
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

class_names = list(train_gen.class_indices.keys())
idx_to_class = {v:k for k,v in train_gen.class_indices.items()}

print("Classes:", class_names)

Found 3788 images belonging to 2 classes.
Found 945 images belonging to 2 classes.
Classes: ['chihuahua', 'muffin']


In [6]:
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0

def build_model(name="efficientnet"):

    try:
        if name == "mobilenet":
            base = MobileNetV2(weights='imagenet', include_top=False,
                               input_shape=(IMG_SIZE, IMG_SIZE, 3))
        else:
            base = EfficientNetB0(weights='imagenet', include_top=False,
                                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
    except:
        print("⚠️ No internet → fallback")
        if name == "mobilenet":
            base = MobileNetV2(weights=None, include_top=False,
                               input_shape=(IMG_SIZE, IMG_SIZE, 3))
        else:
            base = EfficientNetB0(weights=None, include_top=False,
                                 input_shape=(IMG_SIZE, IMG_SIZE, 3))

    base.trainable = False  # freeze first

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)

    output = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    return model

In [7]:
def train_model(model):

    callbacks = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2, factor=0.3),
        ModelCheckpoint("best_model.keras", save_best_only=True)
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=10,
        callbacks=callbacks
    )

    best_acc = max(history.history['val_accuracy'])
    return model, history, best_acc

In [8]:
from tensorflow.keras import layers, models

In [ ]:
models_to_try = ["mobilenet", "efficientnet"]

best_model = None
best_history = None
best_acc = 0
best_name = ""

for name in models_to_try:
    print(f"\n🚀 Training {name}")

    model = build_model(name)
    model, history, acc = train_model(model)

    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_history = history
        best_name = name

print(f"\n🏆 BEST MODEL: {best_name} ({best_acc:.4f})")


🚀 Training mobilenet
⚠️ No internet → fallback


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
 59/237 ━━━━━━━━━━━━━━━━━━━━ 1:10 396ms/step - accuracy: 0.5237 - loss: 0.6943

In [ ]:
# Unfreeze last layers
for layer in best_model.layers[-30:]:
    layer.trainable = True

best_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

best_model.fit(train_gen, validation_data=val_gen, epochs=5)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(best_history.history['accuracy'], label='Train')
plt.plot(best_history.history['val_accuracy'], label='Val')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(best_history.history['loss'], label='Train')
plt.plot(best_history.history['val_loss'], label='Val')
plt.title("Loss")
plt.legend()

plt.show()

In [ ]:
results = []

for img_name in os.listdir(test_dir):
    img_path = os.path.join(test_dir, img_name)

    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prob = best_model.predict(img_array, verbose=0)[0][0]

    label = idx_to_class[int(prob > 0.5)]

    results.append([img_name, label])

submission = pd.DataFrame(results, columns=["ID", "Label"])
submission.to_csv("submission.csv", index=False)

print("✅ submission.csv ready!")